In [2]:
import pandas as pd
# from classifiers import *
# import accelerate
# import gc
# import torch
import os
import argparse
import pickle

In [3]:
df_frames_test = pd.read_csv('labelled_frames_test.csv')
df_frames_train = pd.read_csv('labelled_frames_train.csv')

In [4]:
frames = df_frames_test.columns[-7:]
print(df_frames_test.columns)
print(frames)

Index(['stories_id', 'publish_date', 'media_name', 'title', 'url', 'text',
       'keywords', 'sexual stigma and transmission routes',
       'racial disparities and stigmatising name', 'global relations',
       'public health failure', 'epidemic preparedness and surveillance',
       'human-interest stories', 'broader health issues'],
      dtype='object')
Index(['sexual stigma and transmission routes',
       'racial disparities and stigmatising name', 'global relations',
       'public health failure', 'epidemic preparedness and surveillance',
       'human-interest stories', 'broader health issues'],
      dtype='object')


In [14]:
from sklearn.model_selection import train_test_split

def per_frame_counts(y_train: pd.DataFrame, y_val: pd.DataFrame, y_test: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a tidy table with per-frame counts across splits:
      columns: [split, frame, n, n_pos, n_neg, pos_rate, neg_rate, imbalance_ratio]
    imbalance_ratio = max(n_pos, n_neg) / max(1, min(n_pos, n_neg))
    """
    def one_split(df, split_name):
        rows = []
        for frame in df.columns:
            n = len(df)
            n_pos = int(df[frame].sum())
            n_neg = int(n - n_pos)
            pos_rate = n_pos / n if n else 0.0
            neg_rate = n_neg / n if n else 0.0
            denom = max(1, min(n_pos, n_neg))
            imb = (max(n_pos, n_neg) / denom) if n else 0.0
            rows.append({
                "split": split_name,
                "frame": frame,
                "n": n,
                "n_pos": n_pos,
                "n_neg": n_neg,
                "pos_rate": round(pos_rate, 4),
                "neg_rate": round(neg_rate, 4),
                "imbalance_ratio": round(imb, 2),
            })
        return pd.DataFrame(rows)

    out = pd.concat([
        one_split(y_train, "train"),
        one_split(y_val,   "val"),
        one_split(y_test,  "test"),
    ], ignore_index=True)
    return out

cols_for_x = ['title', 'text']
X_test_NB = df_frames_test['text']
X_test_BERT = df_frames_test[cols_for_x]
y_test = df_frames_test[frames]


X_train_BERT, X_val_BERT, y_train, y_val = train_test_split(
        df_frames_train[cols_for_x], 
        df_frames_train[frames], 
        test_size=1.0 - (300 / len(df_frames_train)),
        random_state=5,
    )

# Example usage inside your script (after you have y_train, y_val, y_test, frames):
counts_df = per_frame_counts(y_train[frames], y_val[frames], y_test[frames])
print(counts_df.sort_values(["split","frame"]).to_string(index=False))
# counts_df.to_csv(f"{SAVE_DIR}/per_frame_label_counts.csv", index=False)


split                                    frame   n  n_pos  n_neg  pos_rate  neg_rate  imbalance_ratio
 test                    broader health issues 100     21     79    0.2100    0.7900             3.76
 test   epidemic preparedness and surveillance 100     31     69    0.3100    0.6900             2.23
 test                         global relations 100     10     90    0.1000    0.9000             9.00
 test                   human-interest stories 100      9     91    0.0900    0.9100            10.11
 test                    public health failure 100     19     81    0.1900    0.8100             4.26
 test racial disparities and stigmatising name 100     12     88    0.1200    0.8800             7.33
 test    sexual stigma and transmission routes 100     38     62    0.3800    0.6200             1.63
train                    broader health issues 300     62    238    0.2067    0.7933             3.84
train   epidemic preparedness and surveillance 300     75    225    0.2500    0.75